# End-to-End Evaluation Example

This notebook demonstrates the complete workflow for:
1. Cloning a test repository
2. Mining a PR with linked issues
3. Creating a review case
4. Setting up the runner to replay the fix

We use `ai4curation/issue-pr-test-repo` - a small, frozen repo designed for testing.

## Setup

First, let's set up our test repository and constants.

In [ ]:
import subprocess
import shutil
from pathlib import Path

# Test repository constants
TEST_REPO = "ai4curation/issue-pr-test-repo"
TEST_ISSUE = 10  # "create a file poem.md"
TEST_PR = 11     # "Add dragon poem" - closes #10, has 3 commits

# Output directory for this example
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Test repo: {TEST_REPO}")
print(f"Issue: #{TEST_ISSUE}")
print(f"PR: #{TEST_PR}")

## Step 1: Clone the Test Repository

In [ ]:
WORKTREE = OUTPUT_DIR / "issue-pr-test-repo"

# Clean up any existing clone
if WORKTREE.exists():
    shutil.rmtree(WORKTREE)

# Clone the repository
result = subprocess.run(
    ["git", "clone", f"https://github.com/{TEST_REPO}.git", str(WORKTREE)],
    capture_output=True,
    text=True,
)

if result.returncode == 0:
    print(f"Successfully cloned to {WORKTREE}")
else:
    print(f"Error: {result.stderr}")

## Step 2: Mine the PR

Use `mine_pr` to extract rich data from PR #11.

In [ ]:
from ai4c_scribe.pr_mining import mine_pr, build_issue_pr_graph

# Build issue-PR mapping (optional but recommended)
issue_pr_graph = build_issue_pr_graph(TEST_REPO, [TEST_PR])
print(f"Issue-PR graph: {issue_pr_graph}")

In [ ]:
# Mine the PR
record = mine_pr(TEST_REPO, TEST_PR, issue_pr_graph)

print(f"PR #{record.pr_number}: {record.metadata.title}")
print(f"Category: {record.category}")
print(f"Total commits: {record.commits.total_commits}")
print(f"Reviews: {record.reviews.review_count}")
print(f"Linked issues: {[i.number for i in record.linked_issues.issues]}")

### Inspect the Commits

In [ ]:
for commit in record.commits.commits:
    print(f"\n{commit.sha[:8]}: {commit.message_headline}")
    print(f"  Author: {commit.author}")
    print(f"  Files changed: {commit.files_changed}")

### Inspect the Linked Issue

In [ ]:
if record.linked_issues.issues:
    issue = record.linked_issues.issues[0]
    print(f"Issue #{issue.number}: {issue.title}")
    print(f"\nBody:\n{issue.body}")
    print(f"\nLabels: {issue.labels}")
else:
    print("No linked issues found")

## Step 3: Create a Review Case

Review cases capture the state at "first revision" - useful for training LLMs on code review.

In [ ]:
from ai4c_scribe.pr_mining import create_review_case_from_record

review_case = create_review_case_from_record(record)

if review_case:
    print(f"Review Case for PR #{review_case.pr_number}")
    print(f"Parent commit: {review_case.parent_commit_sha[:8]}")
    print(f"First revision action: {review_case.first_revision_action}")
    print(f"Reviews in first revision: {review_case.num_reviews_in_first_revision}")
else:
    print("No review case created (PR may have no reviews)")

## Step 4: Set Up the Runner

The runner prepares the worktree to replay fixing an issue.

In [ ]:
from ai4c_scribe.runner import (
    get_issue_context,
    find_pr_for_issue,
    get_checkout_sha,
    reset_worktree,
    create_branch,
)

### Get Issue Context

In [ ]:
issue_context = get_issue_context(TEST_REPO, TEST_ISSUE)

print(f"Issue #{issue_context.number}: {issue_context.title}")
print(f"Author: @{issue_context.author}")
print(f"Labels: {issue_context.labels}")
print(f"\nBody:\n{issue_context.body}")
print(f"\nComments: {len(issue_context.comments)}")

### Find Linked PR and Checkout SHA

In [ ]:
# Find the PR that fixes this issue
pr_number = find_pr_for_issue(TEST_REPO, TEST_ISSUE)
print(f"Found PR #{pr_number} that references issue #{TEST_ISSUE}")

# Get the checkout SHA (parent of first PR commit)
checkout_sha = get_checkout_sha(TEST_REPO, pr_number, WORKTREE)
print(f"Checkout SHA (parent of first commit): {checkout_sha[:8]}...")

### Reset Worktree and Create Branch

In [ ]:
# Reset to the state before the PR
reset_worktree(WORKTREE, checkout_sha)

# Check current HEAD
result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=WORKTREE,
    capture_output=True,
    text=True,
)
print(f"Current HEAD: {result.stdout.strip()[:8]}...")

# Create experiment branch
experiment_id = "eval-001"
branch_name = create_branch(WORKTREE, experiment_id, TEST_ISSUE)
print(f"Created branch: {branch_name}")

### Verify Worktree State

In [ ]:
# Check current branch
result = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=WORKTREE,
    capture_output=True,
    text=True,
)
print(f"Current branch: {result.stdout.strip()}")

# List files in worktree
print("\nFiles in worktree:")
for f in sorted(WORKTREE.iterdir()):
    if not f.name.startswith('.'):
        print(f"  {f.name}")

### Build Task Instructions

This shows what instructions would be sent to the agent.

In [ ]:
from ai4c_scribe.runner import RunnerConfig, build_task_instructions

config = RunnerConfig(
    experiment_id=experiment_id,
    system_prompt="You are a helpful coding assistant. Follow the issue instructions carefully.",
)

instructions = build_task_instructions(
    issue_context,
    config,
    WORKTREE,
    TEST_REPO,
)

print(instructions)

## Step 5: Compare with Ground Truth

After the agent runs, you can compare its output with the actual PR.

In [ ]:
# Show the diff from the actual PR
print("Ground truth (PR #11 final diff):")
print("=" * 50)
if record.diff:
    print(record.diff.final_diff[:2000] if record.diff.final_diff else "No diff available")
else:
    print("No diff information available")

## Summary

This notebook demonstrated:

1. **Cloning** - Set up a test repository
2. **Mining** - Extract PR data with `mine_pr()`
3. **Review Cases** - Create training data with `create_review_case_from_record()`
4. **Runner Setup** - Prepare worktree to replay a fix:
   - `get_issue_context()` - Fetch issue + comments
   - `find_pr_for_issue()` - Find linked PR
   - `get_checkout_sha()` - Get parent of first PR commit
   - `reset_worktree()` - Reset to pre-PR state
   - `create_branch()` - Create experiment branch
   - `build_task_instructions()` - Generate agent prompt

The agent would then fix the issue, and you can compare its solution with the ground truth PR.

## Cleanup

In [ ]:
# Uncomment to clean up
# shutil.rmtree(WORKTREE)
# print(f"Removed {WORKTREE}")